In [1]:
# ─────────────────────────────────────────────# Imports & dependency installation (run this cell first)# ─────────────────────────────────────────────import subprocesspackages = [    "bitsandbytes>=0.46.1",    "transformers>=4.40.0",    "peft>=0.10.0",    "trl>=0.8.0",    "accelerate>=0.27.0",    "datasets>=2.18.0",]for pkg in packages:    subprocess.run(["pip", "install", "-q", pkg], check=True)import csvimport jsonimport osimport refrom dataclasses import dataclass, fieldfrom typing import Optionalimport torchfrom datasets import Datasetfrom peft import LoraConfig, get_peft_model, prepare_model_for_kbit_trainingfrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfigfrom tqdm import tqdmfrom trl import SFTTrainer, SFTConfigprint("Imports and dependencies ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.5 MB/s eta 0:00:00


ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
# ─────────────────────────────────────────────# Data Structures# ─────────────────────────────────────────────@dataclassclass Question:    """Represents a single question/event sample from questions.jsonl"""    id: str  # e.g. "q-1" — required for Codabench submission    topic_id: int  # links to docs.json; int in dataset    target_event: str    option_a: str    option_b: str    option_c: str    option_d: str    golden_answer: Optional[list[str]] = None  # None for test split (no labels)    @property    def options(self) -> dict:        return {            "A": self.option_a,            "B": self.option_b,            "C": self.option_c,            "D": self.option_d,        }@dataclassclass Document:    """Represents a single contextual document from docs.json"""    doc_id: str    title: str    text: str    source: Optional[str] = None@dataclassclass Sample:    """A fully linked sample: question + its context documents"""    question: Question    documents: list[Document] = field(default_factory=list)# ─────────────────────────────────────────────# Loader Functions# ─────────────────────────────────────────────def load_questions(filepath: str) -> list[Question]:    """Load questions from a .jsonl file (one JSON object per line)."""    questions = []    with open(filepath, "r", encoding="utf-8") as f:        for line in f:            line = line.strip()            if not line:                continue            obj = json.loads(line)            # Golden answer may be missing in test data            golden = obj.get("golden_answer", None)            # Normalize to list if it's a string like "A" or "A,B"            if isinstance(golden, str):                golden = [g.strip() for g in golden.split(",")]            q = Question(                id=obj["id"],                topic_id=obj["topic_id"],                target_event=obj["target_event"],                option_a=obj["option_A"],                option_b=obj["option_B"],                option_c=obj["option_C"],                option_d=obj["option_D"],                golden_answer=golden,            )            questions.append(q)    return questionsdef load_docs(filepath: str) -> dict[int, list[Document]]:    """    Load documents from docs.json.    docs.json is a list of records: [ {"topic_id": 1, "topic": "...", "docs": [...]}, ... ]    Returns a dict mapping topic_id -> list of Document objects.    """    with open(filepath, "r", encoding="utf-8") as f:        raw = json.load(f)    docs_by_topic: dict[int, list[Document]] = {}    for record in raw:        topic_id = record["topic_id"]        doc_list = record["docs"]        docs = []        for d in doc_list:            doc = Document(                doc_id=d.get("doc_id", d.get("id", "")),                title=d.get("title", ""),                text=d.get("text", d.get("content", "")),                source=d.get("source", None),            )            docs.append(doc)        docs_by_topic[topic_id] = docs    return docs_by_topicdef load_split(split_dir: str) -> list[Sample]:    """    Load a full data split (train/dev/test/sample) from its directory.    Expects: questions.jsonl and docs.json inside split_dir.    Returns a list of fully linked Sample objects.    """    questions_path = os.path.join(split_dir, "questions.jsonl")    docs_path = os.path.join(split_dir, "docs.json")    questions = load_questions(questions_path)    docs_by_topic = load_docs(docs_path)    samples = []    for q in questions:        docs = docs_by_topic.get(q.topic_id, [])        samples.append(Sample(question=q, documents=docs))    return samples# ─────────────────────────────────────────────# Main: Load All Splits# ─────────────────────────────────────────────def load_all_splits(base_dir: str) -> dict[str, list[Sample]]:    """Load train, dev, test, and sample splits from the dataset root directory."""    splits = {        "train": "train_data",        "dev":   "dev_data",        "test":  "test_data",        "sample": "sample_data",    }    all_data = {}    for split_name, folder in splits.items():        split_path = os.path.join(base_dir, folder)        if os.path.isdir(split_path):            all_data[split_name] = load_split(split_path)            print(f"[✓] Loaded '{split_name}': {len(all_data[split_name])} samples")        else:            print(f"[!] Skipped '{split_name}': folder not found at {split_path}")    return all_data# ─────────────────────────────────────────────# Quick Inspection Helper# ─────────────────────────────────────────────def inspect_sample(sample: Sample):    """Print a human-readable summary of a single sample."""    q = sample.question    print("=" * 60)    print(f"ID           : {q.id}")    print(f"Topic ID     : {q.topic_id}")    print(f"Target Event : {q.target_event}")    print(f"Options:")    for key, val in q.options.items():        print(f"  [{key}] {val}")    print(f"Golden Answer: {q.golden_answer}")    print(f"Docs linked  : {len(sample.documents)}")    if sample.documents:        first_doc = sample.documents[0]        print(f"  First doc title : {first_doc.title}")        print(f"  First doc text  : {first_doc.text[:200]}...")    print("=" * 60)# ─────────────────────────────────────────────# Entry Point# ─────────────────────────────────────────────if __name__ == "__main__":    # ← Update this path to your local dataset root    # BASE_DIR = "semeval2026-task12-dataset-main"    BASE_DIR = "/kaggle/input/datasets/nosadeghob/semeval2026-task12-dataset-main/semeval2026-task12-dataset-main"    print("Loading dataset...\n")    data = load_all_splits(BASE_DIR)    # Inspect the first training sample as a sanity check    if "train" in data and data["train"]:        print("\n--- Sample Inspection (first train item) ---")        inspect_sample(data["train"][0])    # Summary statistics    print("\n--- Dataset Summary ---")    for split, samples in data.items():        has_labels = sum(1 for s in samples if s.question.golden_answer is not None)        print(f"{split:8s}: {len(samples):4d} samples | {has_labels:4d} labeled")